# Data Quality Monitoring System for E-Commerce Operations

## Project Overview

This project presents the development of a Data Quality Monitoring System using the **Olist Brazilian E-Commerce Public Dataset**. The objective is to simulate a real-world business analytics workflow by identifying, assessing, and improving the quality of transactional data before it is used for reporting and decision-making.

The project follows an end-to-end data analytics pipeline, beginning with data preparation in **Python**, followed by data modelling and quality analysis in **PostgreSQL**, and concluding with interactive **Power BI** dashboards that monitor key data quality metrics and business performance indicators.

Throughout the project, data quality dimensions such as **completeness, accuracy, consistency, validity, uniqueness, and timeliness** are evaluated. The datasets are cleaned, standardized, validated, and transformed into analysis-ready data to support reliable business intelligence and operational reporting.

This project demonstrates practical skills in data cleaning, exploratory data analysis, SQL development, data quality assessment, and dashboard development while following industry-standard data analytics practices.

This script focuses on the preparation of the Olist sellers dataset as part of the Data Quality Monitoring System for E-Commerce Operations. The cleaned dataset supports seller and marketplace analysis by providing seller location information linked to merchants operating on the Olist platform. Preparing this dataset ensures that seller information is complete, consistent, and reliable for downstream analysis in PostgreSQL, SQL, and Power BI.

### **Import The Libraries**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re 
import psycopg2
from sqlalchemy import create_engine 
from pathlib import Path

### **Load The Dataset**

In [2]:
sellers_df = pd.read_csv(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_datasets\raw\olist_sellers_dataset.csv")

## **SELLERS DATASET**

### **1. Data Inspection**

In [3]:
# The first five rows of the sellers dataset
sellers_df.head()

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP


In [4]:
# The number of rows and columns in the sellers dataset
sellers_df.shape

(3095, 4)

In [5]:
# The names and data types of the columns in the sellers dataset
sellers_df.dtypes

seller_id                 object
seller_zip_code_prefix     int64
seller_city               object
seller_state              object
dtype: object

The data types assigned to the Sellers dataset are appropriate for the majority of the variables. The **seller_id**, **seller_city**, and **seller_state** columns are correctly stored as text, reflecting their roles as an identifier and categorical location attributes. However, **seller_zip_code_prefix** is currently stored as an integer despite representing a geographic identifier rather than a numerical value intended for mathematical calculations. To maintain consistency with the data type conventions established in the **Customers** and **Geolocation** datasets, and to preserve ZIP code formatting where leading zeros may occur, **seller_zip_code_prefix** will be converted to a string data type during the cleaning stage. 

In [6]:
# The number of missing values in each column of the sellers dataset
sellers_df.isnull().sum()

seller_id                 0
seller_zip_code_prefix    0
seller_city               0
seller_state              0
dtype: int64

No missing values were identified in the **seller_id**, **seller_zip_code_prefix**, **seller_city**, or **seller_state** columns. All **3,095** seller records contain complete identifier and location information, ensuring that each seller can be accurately linked to related tables within the Olist database.

In [7]:
# The number of duplicates in the sellers dataset 
print(sellers_df.duplicated().sum())

0


In [8]:
# The number of duplicates in each column of the sellers dataset 
for col in sellers_df.columns:
    duplicates = sellers_df[col].duplicated().sum() 
    print(f"{col}: {duplicates}")

seller_id: 0
seller_zip_code_prefix: 849
seller_city: 2484
seller_state: 3072


No fully duplicated records were identified in the Sellers dataset, confirming that each record represents a unique seller. Although duplicate values were observed within the **seller_zip_code_prefix**, **seller_city**, and **seller_state** columns, these repetitions are expected because multiple sellers can operate within the same postal area, city, or state. Furthermore, **seller_id** contains **3,095 unique values**, confirming that every seller is uniquely identifiable.

In [9]:
# The summary statistics of the sellers dataset 
sellers_df.describe()

,seller_zip_code_prefix
count,3095.000000
mean,32291.059451
std,32713.453830
min,1001.000000
25%,7093.500000
50%,14940.000000
75%,64552.500000
max,99730.000000


In [10]:
# The summary statistics of categorical columns in the sellers dataset
sellers_df.describe(include=[object])

,seller_id,seller_city,seller_state
count,3095,3095,3095
unique,3095,611,23
top,3442f8959a84dea7ee197c632cb2df15,sao paulo,SP
freq,1,694,1849


The summary statistics indicate that each seller is uniquely identified by **seller_id**, while seller locations are distributed across **611 cities** and **23 states**, with **Sao Paulo (SP)** representing the largest concentration of sellers. This geographic distribution is consistent with patterns observed in previously prepared Olist datasets, reinforcing the consistency of the integrated database. However, **seller_zip_code_prefix** is currently treated as a numerical variable because it is stored as an integer, resulting in descriptive statistics such as the mean and standard deviation that are not meaningful for a geographic identifier. To maintain consistency with the **Customers** and **Geolocation** datasets and preserve ZIP code formatting, **seller_zip_code_prefix** will be converted to a string data type during the cleaning stage.

### **2. Data Profiling**

#### 2.1. Duplicate Records

In [11]:
# Check for fully duplicated records
duplicate_rows = sellers_df.duplicated().sum()

print(f"Number of fully duplicated records: {duplicate_rows}")

Number of fully duplicated records: 0


In [12]:
# Check for duplicate values in each column
duplicate_summary = pd.DataFrame({
    "Duplicate Count": sellers_df.apply(lambda column: column.duplicated().sum()),
    "Duplicate Percentage": (
        sellers_df.apply(lambda column: column.duplicated().sum()) / len(sellers_df) * 100
    ).round(2)
})

duplicate_summary

,Duplicate Count,Duplicate Percentage
seller_id,0,0.00
seller_zip_code_prefix,849,27.43
seller_city,2484,80.26
seller_state,3072,99.26


In [13]:
# Verify that seller_id is unique
duplicate_seller_ids = sellers_df["seller_id"].duplicated().sum()

print(f"Duplicate seller_id values: {duplicate_seller_ids}")

if duplicate_seller_ids == 0:
    print("seller_id is unique.")
else:
    print("Duplicate seller_id values detected.")

Duplicate seller_id values: 0
seller_id is unique.


No fully duplicated records were identified in the Sellers dataset, confirming that each record represents a unique seller. Verification of the **seller_id** column further confirmed that all **3,095** seller identifiers are unique, satisfying the primary key requirement and supporting reliable relationships with other tables in the integrated Olist database. Although duplicate values were observed in the **seller_zip_code_prefix** (**849**, **27.43%**), **seller_city** (**2,484**, **80.26%**), and **seller_state** (**3,072**, **99.26%**) columns, these repetitions are expected because multiple sellers may operate within the same geographic locations. Consequently, these duplicates do not represent data quality issues.

#### 2.2. Seller ID Validation

In [14]:
# Check the length of each seller_id
seller_id_length = sellers_df["seller_id"].str.len()

seller_id_length.describe()

count    3095.0
mean       32.0
std         0.0
min        32.0
25%        32.0
50%        32.0
75%        32.0
max        32.0
Name: seller_id, dtype: float64

In [15]:
# Check the unique lengths of seller_id values
sellers_df["seller_id"].str.len().value_counts().sort_index()

seller_id
32    3095
Name: count, dtype: int64

In [16]:
# Check for seller_id values containing invalid characters
invalid_seller_ids = sellers_df[
    ~sellers_df["seller_id"].str.fullmatch(r"[a-f0-9]+")
]

print(invalid_seller_ids.shape[0])

0


In [17]:
# Check for leading or trailing whitespace in seller_id
seller_id_whitespace = sellers_df[
    sellers_df["seller_id"] != sellers_df["seller_id"].str.strip()
]

print(seller_id_whitespace.shape[0])

0


All **3,095** seller identifiers follow a consistent **32-character** format with no variation in length. Additionally, no identifiers contained invalid characters outside the expected lowercase hexadecimal format, and no leading or trailing whitespace was detected.

#### 2.3. ZIP Code Prefix

In [18]:
# Check the length of seller_zip_code_prefix values
seller_zip_length = sellers_df["seller_zip_code_prefix"].astype(str).str.len()

In [19]:
# Check the frequency of ZIP code prefix lengths
seller_zip_length.value_counts().sort_index()

seller_zip_code_prefix
4    1027
5    2068
Name: count, dtype: int64

In [20]:
# Identify ZIP code prefixes shorter than five digits
short_zip_prefixes = sellers_df[
    sellers_df["seller_zip_code_prefix"].astype(str).str.len() < 5
]

print(f"ZIP code prefixes shorter than 5 digits: {len(short_zip_prefixes)}")

short_zip_prefixes.head()

ZIP code prefixes shorter than 5 digits: 1027


,seller_id,seller_zip_code_prefix,seller_city,seller_state
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
8,768a86e36ad6aae3d03ee3c6433d61df,1529,sao paulo,SP
12,8bd0f31cf0a614c658f6763bd02dea69,1222,sao paulo,SP
13,05a48cc8859962767935ab9087417fbb,5372,sao paulo,SP
19,f9ec7093df3a7b346b7bcf7864069ca3,5138,sao paulo,SP


In [21]:
# Check for ZIP code prefixes containing non-numeric characters
invalid_zip_prefixes = sellers_df[
    ~sellers_df["seller_zip_code_prefix"].astype(str).str.fullmatch(r"\d+")
]

print(f"ZIP code prefixes containing non-numeric characters: {len(invalid_zip_prefixes)}")

ZIP code prefixes containing non-numeric characters: 0


 The validation identified **2,068** records with the expected five-digit ZIP code prefix format and **1,027** records stored as four-digit values. This pattern is consistent with the **Customers** and **Geolocation** datasets and indicates that leading zeros have been omitted due to the column being stored as an integer rather than a text field. No ZIP code prefixes containing non-numeric characters were identified, confirming that the underlying values remain valid. To preserve the original ZIP code formatting and maintain consistency across the integrated Olist database, **seller_zip_code_prefix** will be converted to a string data type and formatted as a five-character text field during the cleaning stage.

#### 2.4. Seller_city Anomaly

##### 2.4.1 Leading and Trailing Whitespace

In [22]:
# Check for leading or trailing whitespace in seller city
city_whitespace = sellers_df[
    sellers_df["seller_city"] != sellers_df["seller_city"].str.strip()
]

print(
    f"Seller city records with leading or trailing whitespace: "
    f"{len(city_whitespace)}"
)

Seller city records with leading or trailing whitespace: 0


##### 2.4.2. Consecutive Whitespace

In [23]:
# Check for consecutive whitespace within seller city names
consecutive_spaces = sellers_df[
    sellers_df["seller_city"].str.contains(
        r"\s{2,}",
        regex=True,
        na=False
    )
]

print(
    f"Seller city records with consecutive whitespace: "
    f"{len(consecutive_spaces)}"
)

consecutive_spaces[
    ["seller_id", "seller_city", "seller_state"]
].head(20)

Seller city records with consecutive whitespace: 3


,seller_id,seller_city,seller_state
191,e88165a185134e13fdfc85d4fa654db8,ferraz de vasconcelos,SP
576,8a1ff5c35f6595a73fef4c7b96e4908a,sao jose dos pinhais,PR
1705,ea566164622c6b439516ab18062c42cd,sao paulo,SP


The **seller_city** field contains **3 records with consecutive whitespace**. These formatting inconsistencies may result in multiple textual representations of the same city and could affect grouping, matching, and comparisons with geographic reference data. Although the affected records represent a very small proportion of the **3,095 seller records**, the inconsistency should be corrected to maintain standardized city values across the integrated dataset.

##### 2.4.3 Forward and Backward Slashes

In [24]:
# Check for forward or backward slashes in seller city names
slash_cities = sellers_df[
    sellers_df["seller_city"].str.contains(
        r"[/\\]",
        regex=True,
        na=False
    )
]

print(
    f"Seller city records containing forward or backward slashes: "
    f"{len(slash_cities)}"
)

slash_cities[
    ["seller_id", "seller_city", "seller_state"]
]

Seller city records containing forward or backward slashes: 16


,seller_id,seller_city,seller_state
237,c3aad7dc65449ae90a5e9c3c6c1e78e0,auriflama/sp,SP
246,71593c7413973a1e160057b80d4958f6,sao paulo / sao paulo,SP
622,7994b065a7ffb14e71c6312cf87b9de2,cariacica / es,ES
869,cbf09e831b0c11f6f23ffb51004db972,sbc/sp,SP
945,f52c2422904463fdd7741f99045fecb6,santo andre/sao paulo,SP
1004,1cbd32d00d01bb8087a5eb088612fd9c,sp / sp,SP
1159,89dda63a3c907c468ec88c310ed91213,maua/sao paulo,SP
1337,720e6cf846ea7572cbb66b743fb91e6c,mogi das cruzes / sp,SP
1346,cf1313c6e2c01c2f4b014f97db4bcd2b,rio de janeiro \rio de janeiro,RJ
1447,fe9d9cf8631285d5982c6e2cf27fb114,barbacena/ minas gerais,MG


The **seller_city** field contains **16 records with forward or backward slash characters**. Inspection shows that several values contain additional geographic information, including embedded state abbreviations or state names, while others contain repeated or combined city information. For example, values such as **`cariacica / es`**, **`mogi das cruzes / sp`**, and **`santo andre/sao paulo`** indicate that the field is not consistently limited to the seller's city.

In [25]:
# Split slash-containing city values into components
slash_analysis = slash_cities[
    ["seller_id", "seller_city", "seller_state"]
].copy()

slash_analysis["city_before_slash"] = (
    slash_analysis["seller_city"]
    .str.split(r"[/\\]", regex=True)
    .str[0]
    .str.strip()
)

slash_analysis["value_after_slash"] = (
    slash_analysis["seller_city"]
    .str.split(r"[/\\]", regex=True)
    .str[1]
    .str.strip()
)

slash_analysis

,seller_id,seller_city,seller_state,city_before_slash,value_after_slash
237,c3aad7dc65449ae90a5e9c3c6c1e78e0,auriflama/sp,SP,auriflama,sp
246,71593c7413973a1e160057b80d4958f6,sao paulo / sao paulo,SP,sao paulo,sao paulo
622,7994b065a7ffb14e71c6312cf87b9de2,cariacica / es,ES,cariacica,es
869,cbf09e831b0c11f6f23ffb51004db972,sbc/sp,SP,sbc,sp
945,f52c2422904463fdd7741f99045fecb6,santo andre/sao paulo,SP,santo andre,sao paulo
1004,1cbd32d00d01bb8087a5eb088612fd9c,sp / sp,SP,sp,sp
1159,89dda63a3c907c468ec88c310ed91213,maua/sao paulo,SP,maua,sao paulo
1337,720e6cf846ea7572cbb66b743fb91e6c,mogi das cruzes / sp,SP,mogi das cruzes,sp
1346,cf1313c6e2c01c2f4b014f97db4bcd2b,rio de janeiro \rio de janeiro,RJ,rio de janeiro,rio de janeiro
1447,fe9d9cf8631285d5982c6e2cf27fb114,barbacena/ minas gerais,MG,barbacena,minas gerais


The 16 city records containing forward or backward slashes were split into components to determine whether the value after the slash represented a state, state name, or repeated city value. 

In [26]:
# Compare slash-separated values with seller state
slash_state_comparison = slash_analysis.copy()

slash_state_comparison["after_matches_state"] = (
    slash_state_comparison["value_after_slash"].str.upper()
    == slash_state_comparison["seller_state"].str.upper()
)

slash_state_comparison[
    [
        "seller_city",
        "seller_state",
        "city_before_slash",
        "value_after_slash",
        "after_matches_state"
    ]
]

,seller_city,seller_state,city_before_slash,value_after_slash,after_matches_state
237,auriflama/sp,SP,auriflama,sp,True
246,sao paulo / sao paulo,SP,sao paulo,sao paulo,False
622,cariacica / es,ES,cariacica,es,True
869,sbc/sp,SP,sbc,sp,True
945,santo andre/sao paulo,SP,santo andre,sao paulo,False
1004,sp / sp,SP,sp,sp,True
1159,maua/sao paulo,SP,maua,sao paulo,False
1337,mogi das cruzes / sp,SP,mogi das cruzes,sp,True
1346,rio de janeiro \rio de janeiro,RJ,rio de janeiro,rio de janeiro,False
1447,barbacena/ minas gerais,MG,barbacena,minas gerais,False


The value after the slash was compared with `seller_state` to determine whether it represented the seller's state. The results showed that some records contained state abbreviations, while others contained full state names or repeated city names.

In [27]:
# Brazilian state names for geographic comparison
brazil_states = {
    "AC": "acre",
    "AL": "alagoas",
    "AP": "amapa",
    "AM": "amazonas",
    "BA": "bahia",
    "CE": "ceara",
    "DF": "distrito federal",
    "ES": "espirito santo",
    "GO": "goias",
    "MA": "maranhao",
    "MT": "mato grosso",
    "MS": "mato grosso do sul",
    "MG": "minas gerais",
    "PA": "para",
    "PB": "paraiba",
    "PR": "parana",
    "PE": "pernambuco",
    "PI": "piaui",
    "RJ": "rio de janeiro",
    "RN": "rio grande do norte",
    "RS": "rio grande do sul",
    "RO": "rondonia",
    "RR": "roraima",
    "SC": "santa catarina",
    "SP": "sao paulo",
    "SE": "sergipe",
    "TO": "tocantins"
}

slash_state_comparison["after_is_state_name"] = (
    slash_state_comparison["value_after_slash"]
    .str.lower()
    .isin(brazil_states.values())
)

slash_state_comparison[
    [
        "seller_city",
        "seller_state",
        "value_after_slash",
        "after_matches_state",
        "after_is_state_name"
    ]
]

,seller_city,seller_state,value_after_slash,after_matches_state,after_is_state_name
237,auriflama/sp,SP,sp,True,False
246,sao paulo / sao paulo,SP,sao paulo,False,True
622,cariacica / es,ES,es,True,False
869,sbc/sp,SP,sp,True,False
945,santo andre/sao paulo,SP,sao paulo,False,True
1004,sp / sp,SP,sp,True,False
1159,maua/sao paulo,SP,sao paulo,False,True
1337,mogi das cruzes / sp,SP,sp,True,False
1346,rio de janeiro \rio de janeiro,RJ,rio de janeiro,False,True
1447,barbacena/ minas gerais,MG,minas gerais,False,True


The values after the slash were further checked against known state names to distinguish state information from repeated city values. This confirmed that several records contained redundant state names, while others contained duplicated city names.

In [28]:
# Compare the city component with the seller state
slash_analysis["city_before_slash"] = (
    slash_analysis["city_before_slash"]
    .str.lower()
    .str.strip()
)

slash_analysis[
    [
        "seller_id",
        "city_before_slash",
        "seller_state",
        "seller_city"
    ]
]

,seller_id,city_before_slash,seller_state,seller_city
237,c3aad7dc65449ae90a5e9c3c6c1e78e0,auriflama,SP,auriflama/sp
246,71593c7413973a1e160057b80d4958f6,sao paulo,SP,sao paulo / sao paulo
622,7994b065a7ffb14e71c6312cf87b9de2,cariacica,ES,cariacica / es
869,cbf09e831b0c11f6f23ffb51004db972,sbc,SP,sbc/sp
945,f52c2422904463fdd7741f99045fecb6,santo andre,SP,santo andre/sao paulo
1004,1cbd32d00d01bb8087a5eb088612fd9c,sp,SP,sp / sp
1159,89dda63a3c907c468ec88c310ed91213,maua,SP,maua/sao paulo
1337,720e6cf846ea7572cbb66b743fb91e6c,mogi das cruzes,SP,mogi das cruzes / sp
1346,cf1313c6e2c01c2f4b014f97db4bcd2b,rio de janeiro,RJ,rio de janeiro \rio de janeiro
1447,fe9d9cf8631285d5982c6e2cf27fb114,barbacena,MG,barbacena/ minas gerais


In [29]:
geolocation_df = pd.read_csv (r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_datasets\cleaned\olist_geolocation_cleaned.csv")

In [30]:
# Compare slash-containing seller locations with the cleaned geolocation dataset
geolocation_reference = (
    geolocation_df[
        [
            "geolocation_zip_code_prefix",
            "geolocation_city",
            "geolocation_state"
        ]
    ]
    .drop_duplicates()
)

slash_geolocation_comparison = slash_analysis.merge(
    geolocation_reference,
    left_on=[
        "city_before_slash",
        "seller_state"
    ],
    right_on=[
        "geolocation_city",
        "geolocation_state"
    ],
    how="left",
    indicator=True
)

slash_geolocation_comparison[
    [
        "seller_id",
        "seller_city",
        "seller_state",
        "city_before_slash",
        "geolocation_city",
        "geolocation_state",
        "_merge"
    ]
]

,seller_id,seller_city,seller_state,city_before_slash,geolocation_city,geolocation_state,_merge
0,c3aad7dc65449ae90a5e9c3c6c1e78e0,auriflama/sp,SP,auriflama,auriflama,SP,both
1,71593c7413973a1e160057b80d4958f6,sao paulo / sao paulo,SP,sao paulo,sao paulo,SP,both
2,71593c7413973a1e160057b80d4958f6,sao paulo / sao paulo,SP,sao paulo,sao paulo,SP,both
3,71593c7413973a1e160057b80d4958f6,sao paulo / sao paulo,SP,sao paulo,sao paulo,SP,both
4,71593c7413973a1e160057b80d4958f6,sao paulo / sao paulo,SP,sao paulo,sao paulo,SP,both
...,...,...,...,...,...,...,...
4289,7f5e4d5efad7e44b91115dd1decb65f3,jacarei / sao paulo,SP,jacarei,jacarei,SP,both
4290,7f5e4d5efad7e44b91115dd1decb65f3,jacarei / sao paulo,SP,jacarei,jacarei,SP,both
4291,7f5e4d5efad7e44b91115dd1decb65f3,jacarei / sao paulo,SP,jacarei,jacarei,SP,both
4292,7f5e4d5efad7e44b91115dd1decb65f3,jacarei / sao paulo,SP,jacarei,jacarei,SP,both


The seller ZIP code and state were compared with the cleaned Geolocation dataset to validate the geographic location of the affected records. This confirmed the correct city for the ambiguous cases, including `sbc` as `sao bernardo do campo` and `sp` as `sao paulo`.

In [31]:
# Create a unique city-state reference from the cleaned geolocation dataset
geolocation_city_state = (
    geolocation_df[
        ["geolocation_city", "geolocation_state"]
    ]
    .drop_duplicates()
)

# Check whether the city before the slash exists for the seller's state
slash_city_validation = slash_analysis[
    ["seller_id", "seller_city", "seller_state", "city_before_slash"]
].drop_duplicates().copy()

slash_city_validation = slash_city_validation.merge(
    geolocation_city_state,
    left_on=["city_before_slash", "seller_state"],
    right_on=["geolocation_city", "geolocation_state"],
    how="left",
    indicator=True
)

slash_city_validation[
    [
        "seller_id",
        "seller_city",
        "seller_state",
        "city_before_slash",
        "geolocation_city",
        "geolocation_state",
        "_merge"
    ]
]

,seller_id,seller_city,seller_state,city_before_slash,geolocation_city,geolocation_state,_merge
0,c3aad7dc65449ae90a5e9c3c6c1e78e0,auriflama/sp,SP,auriflama,auriflama,SP,both
1,71593c7413973a1e160057b80d4958f6,sao paulo / sao paulo,SP,sao paulo,sao paulo,SP,both
2,7994b065a7ffb14e71c6312cf87b9de2,cariacica / es,ES,cariacica,cariacica,ES,both
3,cbf09e831b0c11f6f23ffb51004db972,sbc/sp,SP,sbc,NaN,NaN,left_only
4,f52c2422904463fdd7741f99045fecb6,santo andre/sao paulo,SP,santo andre,santo andre,SP,both
5,1cbd32d00d01bb8087a5eb088612fd9c,sp / sp,SP,sp,sp,SP,both
6,89dda63a3c907c468ec88c310ed91213,maua/sao paulo,SP,maua,maua,SP,both
7,720e6cf846ea7572cbb66b743fb91e6c,mogi das cruzes / sp,SP,mogi das cruzes,mogi das cruzes,SP,both
8,cf1313c6e2c01c2f4b014f97db4bcd2b,rio de janeiro \rio de janeiro,RJ,rio de janeiro,rio de janeiro,RJ,both
9,fe9d9cf8631285d5982c6e2cf27fb114,barbacena/ minas gerais,MG,barbacena,barbacena,MG,both


In [32]:
# Compare seller ZIP + state against the geolocation reference

seller_zip_geo = (
    sellers_df[
        [
            "seller_id",
            "seller_city",
            "seller_zip_code_prefix",
            "seller_state"
        ]
    ]
    .loc[sellers_df["seller_id"].isin(slash_analysis["seller_id"])]
    .drop_duplicates()
    .copy()
)

seller_zip_geo["zip_prefix_for_comparison"] = (
    seller_zip_geo["seller_zip_code_prefix"]
    .astype(str)
    .str.zfill(5)
)

zip_city_state_reference = (
    geolocation_df[
        [
            "geolocation_zip_code_prefix",
            "geolocation_city",
            "geolocation_state"
        ]
    ]
    .drop_duplicates()
    .copy()
)

zip_city_state_reference["geolocation_zip_code_prefix"] = (
    zip_city_state_reference["geolocation_zip_code_prefix"]
    .astype(str)
    .str.zfill(5)
)

zip_city_state_check = seller_zip_geo.merge(
    zip_city_state_reference,
    left_on=["zip_prefix_for_comparison", "seller_state"],
    right_on=["geolocation_zip_code_prefix", "geolocation_state"],
    how="left"
)

zip_city_state_check[
    [
        "seller_id",
        "seller_city",
        "seller_zip_code_prefix",
        "seller_state",
        "geolocation_city",
        "geolocation_state"
    ]
]

,seller_id,seller_city,seller_zip_code_prefix,seller_state,geolocation_city,geolocation_state
0,c3aad7dc65449ae90a5e9c3c6c1e78e0,auriflama/sp,15350,SP,auriflama,SP
1,71593c7413973a1e160057b80d4958f6,sao paulo / sao paulo,3407,SP,sao paulo,SP
2,7994b065a7ffb14e71c6312cf87b9de2,cariacica / es,29142,ES,cariacica,ES
3,cbf09e831b0c11f6f23ffb51004db972,sbc/sp,9726,SP,sao bernardo do campo,SP
4,f52c2422904463fdd7741f99045fecb6,santo andre/sao paulo,9230,SP,santo andre,SP
5,1cbd32d00d01bb8087a5eb088612fd9c,sp / sp,3363,SP,sao paulo,SP
6,89dda63a3c907c468ec88c310ed91213,maua/sao paulo,9380,SP,maua,SP
7,720e6cf846ea7572cbb66b743fb91e6c,mogi das cruzes / sp,8717,SP,mogi das cruzes,SP
8,cf1313c6e2c01c2f4b014f97db4bcd2b,rio de janeiro \rio de janeiro,22050,RJ,rio de janeiro,RJ
9,fe9d9cf8631285d5982c6e2cf27fb114,barbacena/ minas gerais,36200,MG,barbacena,MG


##### 2.4.4. Identify Unexpected Special Characters

In [33]:
# Identify seller city values containing characters other than letters and spaces

special_char_pattern = r"[^a-zA-Z\s]"

special_character_cities = sellers_df[
    sellers_df["seller_city"].str.contains(
        special_char_pattern,
        regex=True,
        na=False
    )
]

print(
    "Seller city records containing unexpected special characters:",
    len(special_character_cities)
)

special_character_cities[
    ["seller_id", "seller_city", "seller_state"]
]

Seller city records containing unexpected special characters: 34


,seller_id,seller_city,seller_state
78,731ef20c231d9a7103a425e83fd91271,lages - sc,SC
237,c3aad7dc65449ae90a5e9c3c6c1e78e0,auriflama/sp,SP
246,71593c7413973a1e160057b80d4958f6,sao paulo / sao paulo,SP
360,a3fa18b3f688ec0fca3eb8bfcbd2d5b3,são paulo,SP
476,26b482dccfa29bd2e40703ba45523702,santa barbara d´oeste,SP
517,ceb7b4fb9401cd378de7886317ad1b47,04482255,RJ
551,723a46b89fd5c3ed78ccdf039e33ac63,"novo hamburgo, rio grande do sul, brasil",RS
622,7994b065a7ffb14e71c6312cf87b9de2,cariacica / es,ES
707,11938fcc75f6902fea1d0a6f72b54a57,sao miguel d'oeste,SC
826,05e107217c7266362fd44b75b2cd4cc4,sao paulo - sp,SP


In [34]:
# Extract the unique non-letter/non-space characters found in seller city values

special_characters_found = (
    special_character_cities["seller_city"]
    .str.findall(special_char_pattern)
    .explode()
    .dropna()
    .value_counts()
)

special_characters_found

seller_city
/    15
'     6
-     5
,     4
´     2
4     2
2     2
5     2
.     2
̃     1
0     1
8     1
(     1
)     1
\     1
@     1
Name: count, dtype: int64

In [35]:
# Categorize seller city anomalies by the type of character present

special_character_categories = special_character_cities[
    ["seller_id", "seller_city", "seller_state"]
].copy()

special_character_categories["has_number"] = (
    special_character_categories["seller_city"]
    .str.contains(r"\d", regex=True, na=False)
)

special_character_categories["has_punctuation"] = (
    special_character_categories["seller_city"]
    .str.contains(r"[^\w\s]", regex=True, na=False)
)

special_character_categories["has_non_ascii"] = (
    special_character_categories["seller_city"]
    .str.contains(r"[^\x00-\x7F]", regex=True, na=False)
)

special_character_categories

,seller_id,seller_city,seller_state,has_number,has_punctuation,has_non_ascii
78,731ef20c231d9a7103a425e83fd91271,lages - sc,SC,False,True,False
237,c3aad7dc65449ae90a5e9c3c6c1e78e0,auriflama/sp,SP,False,True,False
246,71593c7413973a1e160057b80d4958f6,sao paulo / sao paulo,SP,False,True,False
360,a3fa18b3f688ec0fca3eb8bfcbd2d5b3,são paulo,SP,False,True,True
476,26b482dccfa29bd2e40703ba45523702,santa barbara d´oeste,SP,False,True,True
517,ceb7b4fb9401cd378de7886317ad1b47,04482255,RJ,True,False,False
551,723a46b89fd5c3ed78ccdf039e33ac63,"novo hamburgo, rio grande do sul, brasil",RS,False,True,False
622,7994b065a7ffb14e71c6312cf87b9de2,cariacica / es,ES,False,True,False
707,11938fcc75f6902fea1d0a6f72b54a57,sao miguel d'oeste,SC,False,True,False
826,05e107217c7266362fd44b75b2cd4cc4,sao paulo - sp,SP,False,True,False


In [36]:
# Display unique seller city values containing unexpected characters

unique_special_character_cities = (
    special_character_cities[
        ["seller_city", "seller_state"]
    ]
    .drop_duplicates()
    .sort_values(["seller_state", "seller_city"])
)

unique_special_character_cities

,seller_city,seller_state
874,arraial d'ajuda (porto seguro),BA
622,cariacica / es,ES
1447,barbacena/ minas gerais,MG
1610,andira-pr,PR
1712,pinhais/pr,PR
2258,vendas@creditparts.com.br,PR
517,04482255,RJ
1649,rio de janeiro / rio de janeiro,RJ
1346,rio de janeiro \rio de janeiro,RJ
2988,"rio de janeiro, rio de janeiro, brasil",RJ


From the results:

- `/` and `\` → already investigated in 1.3. Don't redo them.
- `'` → apostrophe can legitimately occur in names such as D'Oeste.
- ` ´ ` and ` ̃  `→ need to distinguish valid accented characters from malformed Unicode/combining marks.
- `-` → can be legitimate, but these particular values need inspection.
- `,` → suspicious because some values contain city + state/country information.
- `( )` → suspicious because one value contains additional location information.
- Numbers → definitely suspicious for a city field.
- `@` → definitely suspicious; `vendas@creditparts.com.br` is clearly an email address.
- `.` → suspicious when associated with the email/domain value.

In [37]:
# Identify seller city values containing numeric characters

numeric_city_records = sellers_df[
    sellers_df["seller_city"].str.contains(r"\d", regex=True, na=False)
]

print(
    "Seller city records containing numeric characters:",
    len(numeric_city_records)
)

numeric_city_records[
    ["seller_id", "seller_city", "seller_state"]
]

Seller city records containing numeric characters: 1


,seller_id,seller_city,seller_state
517,ceb7b4fb9401cd378de7886317ad1b47,04482255,RJ


In [38]:
# Identify seller city values that appear to contain email addresses

email_city_records = sellers_df[
    sellers_df["seller_city"].str.contains(
        r"@", 
        regex=True,
        na=False
    )
]

print(
    "Seller city records containing email addresses:",
    len(email_city_records)
)

email_city_records[
    ["seller_id", "seller_city", "seller_state"]
]

Seller city records containing email addresses: 1


,seller_id,seller_city,seller_state
2258,4b5f66b7adcf57f1ecc0d3c07dd6b177,vendas@creditparts.com.br,PR


In [39]:
# Identify seller city values containing commas or parentheses

location_punctuation_records = sellers_df[
    sellers_df["seller_city"].str.contains(
        r"[,()]",
        regex=True,
        na=False
    )
]

print(
    "Seller city records containing commas or parentheses:",
    len(location_punctuation_records)
)

location_punctuation_records[
    ["seller_id", "seller_city", "seller_state"]
]

Seller city records containing commas or parentheses: 3


,seller_id,seller_city,seller_state
551,723a46b89fd5c3ed78ccdf039e33ac63,"novo hamburgo, rio grande do sul, brasil",RS
874,4aba391bc3b88717ce08eb11e44937b2,arraial d'ajuda (porto seguro),BA
2988,f9eedec3129e8cc6b6429c42d0808c5b,"rio de janeiro, rio de janeiro, brasil",RJ


In [40]:
# Identify seller city values containing hyphens

hyphen_city_records = sellers_df[
    sellers_df["seller_city"].str.contains(
        r"-",
        regex=True,
        na=False
    )
]

print(
    "Seller city records containing hyphens:",
    len(hyphen_city_records)
)

hyphen_city_records[
    ["seller_id", "seller_city", "seller_state"]
]

Seller city records containing hyphens: 5


,seller_id,seller_city,seller_state
78,731ef20c231d9a7103a425e83fd91271,lages - sc,SC
826,05e107217c7266362fd44b75b2cd4cc4,sao paulo - sp,SP
1580,8090490573c6c0aa343a7231ebcb8c86,sao paulo - sp,SP
1610,20cb7c2fde3e5bf10f0bbe7394e1c6a9,andira-pr,PR
2946,06579cb253ecd5a3a12a9e6eb6bf8f47,sao paulo - sp,SP


In [41]:
# Identify seller city values containing non-ASCII characters

non_ascii_city_records = sellers_df[
    sellers_df["seller_city"].str.contains(
        r"[^\x00-\x7F]",
        regex=True,
        na=False
    )
]

print(
    "Seller city records containing non-ASCII characters:",
    len(non_ascii_city_records)
)

non_ascii_city_records[
    ["seller_id", "seller_city", "seller_state"]
]

Seller city records containing non-ASCII characters: 3


,seller_id,seller_city,seller_state
360,a3fa18b3f688ec0fca3eb8bfcbd2d5b3,são paulo,SP
476,26b482dccfa29bd2e40703ba45523702,santa barbara d´oeste,SP
2340,5dceca129747e92ff8ef7a997dc4f8ca,santa barbara d´oeste,SP


What we found
- **Numeric city values:** 1  `04482255` — clearly invalid.
- **Email-like city values:** 1 → `vendas@creditparts.com.br` — clearly invalid.
- **Comma/parentheses:** 3 → these contain additional geographic information, so they need investigation.
- **Hyphens:** 5 → these appear to be city + state abbreviations, so they need investigation rather than automatic removal.
- **Non-ASCII:** 3 → these are legitimate accented city names:
são paulo
santa barbara d’oeste

So we should not classify all non-ASCII characters as errors.

In [42]:
# Identify seller city values containing apostrophes or accent marks

apostrophe_accent_records = sellers_df[
    sellers_df["seller_city"].str.contains(
        r"['´’]",
        regex=True,
        na=False
    )
]

print(
    "Seller city records containing apostrophe or accent-related characters:",
    len(apostrophe_accent_records)
)

apostrophe_accent_records[
    ["seller_id", "seller_city", "seller_state"]
]

Seller city records containing apostrophe or accent-related characters: 8


,seller_id,seller_city,seller_state
476,26b482dccfa29bd2e40703ba45523702,santa barbara d´oeste,SP
707,11938fcc75f6902fea1d0a6f72b54a57,sao miguel d'oeste,SC
874,4aba391bc3b88717ce08eb11e44937b2,arraial d'ajuda (porto seguro),BA
1890,30a2f535bb48308f991d0b9ad4a8c4bb,santa barbara d'oeste,SP
1919,13dd308f81fea30cc670c656b2b46cc3,santa barbara d'oeste,SP
1982,20f0aeea30bc3b8c4420be8ced4226c0,santa barbara d'oeste,SP
2036,54a1852d1b8f10312c55e906355666ee,santa barbara d'oeste,SP
2340,5dceca129747e92ff8ef7a997dc4f8ca,santa barbara d´oeste,SP


In [43]:
import unicodedata

# Identify seller city values containing combining Unicode characters

combining_character_records = sellers_df[
    sellers_df["seller_city"].apply(
        lambda value: any(
            unicodedata.category(char) == "Mn"
            for char in str(value)
        )
    )
]

print(
    "Seller city records containing combining Unicode characters:",
    len(combining_character_records)
)

combining_character_records[
    ["seller_id", "seller_city", "seller_state"]
]

Seller city records containing combining Unicode characters: 1


,seller_id,seller_city,seller_state
360,a3fa18b3f688ec0fca3eb8bfcbd2d5b3,são paulo,SP


In [44]:
# Inspect the Unicode representation of affected city values

for value in combining_character_records["seller_city"]:
    print(
        repr(value),
        "->",
        [
            (char, unicodedata.name(char, "UNKNOWN"))
            for char in value
        ]
    )

'são paulo' -> [('s', 'LATIN SMALL LETTER S'), ('a', 'LATIN SMALL LETTER A'), ('̃', 'COMBINING TILDE'), ('o', 'LATIN SMALL LETTER O'), (' ', 'SPACE'), ('p', 'LATIN SMALL LETTER P'), ('a', 'LATIN SMALL LETTER A'), ('u', 'LATIN SMALL LETTER U'), ('l', 'LATIN SMALL LETTER L'), ('o', 'LATIN SMALL LETTER O')]


- The apostrophes in `d'oeste` / `d’ajuda` are legitimate parts of the city names.
- The one combining Unicode record is `são paulo`. That is not a different city; it's são paulo represented using a decomposed Unicode character.
- So this is a **Unicode normalization issue**, not a city-name correction.
- We should not clean these manually one by one. The later standardization step can normalize this consistently.

In [45]:
# Combine previously identified non-city-content anomalies for investigation

non_city_anomalies = sellers_df[
    sellers_df["seller_city"].str.contains(
        r"\d|@|[,()]|-",
        regex=True,
        na=False
    )
][
    ["seller_id", "seller_city", "seller_state"]
].drop_duplicates()

print(
    "Unique seller city values requiring non-city-content investigation:",
    len(non_city_anomalies)
)

non_city_anomalies

Unique seller city values requiring non-city-content investigation: 10


,seller_id,seller_city,seller_state
78,731ef20c231d9a7103a425e83fd91271,lages - sc,SC
517,ceb7b4fb9401cd378de7886317ad1b47,04482255,RJ
551,723a46b89fd5c3ed78ccdf039e33ac63,"novo hamburgo, rio grande do sul, brasil",RS
826,05e107217c7266362fd44b75b2cd4cc4,sao paulo - sp,SP
874,4aba391bc3b88717ce08eb11e44937b2,arraial d'ajuda (porto seguro),BA
1580,8090490573c6c0aa343a7231ebcb8c86,sao paulo - sp,SP
1610,20cb7c2fde3e5bf10f0bbe7394e1c6a9,andira-pr,PR
2258,4b5f66b7adcf57f1ecc0d3c07dd6b177,vendas@creditparts.com.br,PR
2946,06579cb253ecd5a3a12a9e6eb6bf8f47,sao paulo - sp,SP
2988,f9eedec3129e8cc6b6429c42d0808c5b,"rio de janeiro, rio de janeiro, brasil",RJ


In [46]:
# Compare suspicious seller city values with the geolocation reference

seller_city_reference = (
    geolocation_df[
        [
            "geolocation_zip_code_prefix",
            "geolocation_city",
            "geolocation_state"
        ]
    ]
    .drop_duplicates()
)

seller_city_check = (
    sellers_df[
        sellers_df["seller_city"].isin(
            non_city_anomalies["seller_city"]
        )
    ][
        [
            "seller_id",
            "seller_zip_code_prefix",
            "seller_city",
            "seller_state"
        ]
    ]
    .copy()
)

seller_city_check["zip_prefix_for_comparison"] = (
    seller_city_check["seller_zip_code_prefix"]
    .astype(str)
    .str.zfill(5)
)

seller_city_reference["geolocation_zip_code_prefix"] = (
    seller_city_reference["geolocation_zip_code_prefix"]
    .astype(str)
    .str.zfill(5)
)

seller_city_check = seller_city_check.merge(
    seller_city_reference,
    left_on=[
        "zip_prefix_for_comparison",
        "seller_state"
    ],
    right_on=[
        "geolocation_zip_code_prefix",
        "geolocation_state"
    ],
    how="left"
)

seller_city_check[
    [
        "seller_id",
        "seller_city",
        "seller_state",
        "geolocation_city",
        "geolocation_state"
    ]
]

,seller_id,seller_city,seller_state,geolocation_city,geolocation_state
0,731ef20c231d9a7103a425e83fd91271,lages - sc,SC,lages,SC
1,ceb7b4fb9401cd378de7886317ad1b47,04482255,RJ,rio de janeiro,RJ
2,723a46b89fd5c3ed78ccdf039e33ac63,"novo hamburgo, rio grande do sul, brasil",RS,novo hamburgo,RS
3,05e107217c7266362fd44b75b2cd4cc4,sao paulo - sp,SP,sao paulo,SP
4,4aba391bc3b88717ce08eb11e44937b2,arraial d'ajuda (porto seguro),BA,arraial d'ajuda,BA
5,4aba391bc3b88717ce08eb11e44937b2,arraial d'ajuda (porto seguro),BA,arraial d ajuda,BA
6,4aba391bc3b88717ce08eb11e44937b2,arraial d'ajuda (porto seguro),BA,porto seguro,BA
7,8090490573c6c0aa343a7231ebcb8c86,sao paulo - sp,SP,sao paulo,SP
8,20cb7c2fde3e5bf10f0bbe7394e1c6a9,andira-pr,PR,barra do jacare,PR
9,4b5f66b7adcf57f1ecc0d3c07dd6b177,vendas@creditparts.com.br,PR,maringa,PR


In [47]:
# Identify suspicious seller city values where the geolocation reference
# provides a matching city for the seller's ZIP code and state

resolvable_city_anomalies = seller_city_check[
    seller_city_check["geolocation_city"].notna()
].copy()

print(
    "Suspicious city records with a matching geolocation city:",
    len(resolvable_city_anomalies)
)

resolvable_city_anomalies[
    [
        "seller_id",
        "seller_city",
        "seller_state",
        "geolocation_city",
        "geolocation_state"
    ]
]

Suspicious city records with a matching geolocation city: 12


,seller_id,seller_city,seller_state,geolocation_city,geolocation_state
0,731ef20c231d9a7103a425e83fd91271,lages - sc,SC,lages,SC
1,ceb7b4fb9401cd378de7886317ad1b47,04482255,RJ,rio de janeiro,RJ
2,723a46b89fd5c3ed78ccdf039e33ac63,"novo hamburgo, rio grande do sul, brasil",RS,novo hamburgo,RS
3,05e107217c7266362fd44b75b2cd4cc4,sao paulo - sp,SP,sao paulo,SP
4,4aba391bc3b88717ce08eb11e44937b2,arraial d'ajuda (porto seguro),BA,arraial d'ajuda,BA
5,4aba391bc3b88717ce08eb11e44937b2,arraial d'ajuda (porto seguro),BA,arraial d ajuda,BA
6,4aba391bc3b88717ce08eb11e44937b2,arraial d'ajuda (porto seguro),BA,porto seguro,BA
7,8090490573c6c0aa343a7231ebcb8c86,sao paulo - sp,SP,sao paulo,SP
8,20cb7c2fde3e5bf10f0bbe7394e1c6a9,andira-pr,PR,barra do jacare,PR
9,4b5f66b7adcf57f1ecc0d3c07dd6b177,vendas@creditparts.com.br,PR,maringa,PR


- **10 unique seller-city values** require further investigation because they contain content that is not a clean city name.
- When compared against the geolocation dataset, **all 10 had at least one matching geolocation city**.
- The merged output produced **12 rows** because some suspicious seller-city values correspond to multiple geolocation-city records.
- 0 suspicious records lacked a geolocation match, so we don't have evidence of completely unrecognised locations here.
- Importantly, the geolocation comparison confirms that these are mostly **format/content anomalies rather than completely invalid locations.**

In [48]:
# Identify suspicious city records without a matching geolocation reference

unresolved_city_anomalies = seller_city_check[
    seller_city_check["geolocation_city"].isna()
].copy()

print(
    "Suspicious city records without a matching geolocation city:",
    len(unresolved_city_anomalies)
)

unresolved_city_anomalies[
    [
        "seller_id",
        "seller_city",
        "seller_state"
    ]
]

Suspicious city records without a matching geolocation city: 0


,seller_id,seller_city,seller_state


In [49]:
# Identify unique seller-city values requiring non-city-content investigation

non_city_content_records = sellers_df[
    sellers_df["seller_city"].str.contains(
        r"\d|@|,|\(|\)|-",
        regex=True,
        na=False
    )
].copy()

# Compare suspicious seller cities against the geolocation reference
city_state_check = non_city_content_records.merge(
    geolocation_df[
        ["geolocation_city", "geolocation_state"]
    ].drop_duplicates(),
    left_on=["seller_city", "seller_state"],
    right_on=["geolocation_city", "geolocation_state"],
    how="left",
    indicator=True
)

city_state_check["city_state_match"] = (
    city_state_check["_merge"] == "both"
)

print(
    "Suspicious records with a matching city-state combination:",
    city_state_check["city_state_match"].sum()
)

print(
    "Suspicious records without a matching city-state combination:",
    (~city_state_check["city_state_match"]).sum()
)

Suspicious records with a matching city-state combination: 0
Suspicious records without a matching city-state combination: 10


In [50]:
# Display suspicious city values without an exact geolocation city-state match

city_state_check[
    ~city_state_check["city_state_match"]
][
    [
        "seller_id",
        "seller_city",
        "seller_state",
        "geolocation_city",
        "geolocation_state"
    ]
]

,seller_id,seller_city,seller_state,geolocation_city,geolocation_state
0,731ef20c231d9a7103a425e83fd91271,lages - sc,SC,NaN,NaN
1,ceb7b4fb9401cd378de7886317ad1b47,04482255,RJ,NaN,NaN
2,723a46b89fd5c3ed78ccdf039e33ac63,"novo hamburgo, rio grande do sul, brasil",RS,NaN,NaN
3,05e107217c7266362fd44b75b2cd4cc4,sao paulo - sp,SP,NaN,NaN
4,4aba391bc3b88717ce08eb11e44937b2,arraial d'ajuda (porto seguro),BA,NaN,NaN
5,8090490573c6c0aa343a7231ebcb8c86,sao paulo - sp,SP,NaN,NaN
6,20cb7c2fde3e5bf10f0bbe7394e1c6a9,andira-pr,PR,NaN,NaN
7,4b5f66b7adcf57f1ecc0d3c07dd6b177,vendas@creditparts.com.br,PR,NaN,NaN
8,06579cb253ecd5a3a12a9e6eb6bf8f47,sao paulo - sp,SP,NaN,NaN
9,f9eedec3129e8cc6b6429c42d0808c5b,"rio de janeiro, rio de janeiro, brasil",RJ,NaN,NaN


In [51]:
# Extract the likely city component from suspicious seller-city values

city_component_check = non_city_content_records.copy()

city_component_check["city_component"] = (
    city_component_check["seller_city"]
    .str.split(r"[-,@()]")
    .str[0]
    .str.strip()
)

city_component_check = city_component_check.merge(
    geolocation_df[
        ["geolocation_city", "geolocation_state"]
    ].drop_duplicates(),
    left_on=["city_component", "seller_state"],
    right_on=["geolocation_city", "geolocation_state"],
    how="left",
    indicator=True
)

city_component_check["city_component_match"] = (
    city_component_check["_merge"] == "both"
)

print(
    "Suspicious records with a matching city component:",
    city_component_check["city_component_match"].sum()
)

print(
    "Suspicious records without a matching city component:",
    (~city_component_check["city_component_match"]).sum()
)

Suspicious records with a matching city component: 8
Suspicious records without a matching city component: 2


In [52]:
city_component_check[
    [
        "seller_id",
        "seller_city",
        "seller_state",
        "city_component",
        "geolocation_city",
        "geolocation_state",
        "city_component_match"
    ]
]

,seller_id,seller_city,seller_state,city_component,geolocation_city,geolocation_state,city_component_match
0,731ef20c231d9a7103a425e83fd91271,lages - sc,SC,lages,lages,SC,True
1,ceb7b4fb9401cd378de7886317ad1b47,04482255,RJ,04482255,NaN,NaN,False
2,723a46b89fd5c3ed78ccdf039e33ac63,"novo hamburgo, rio grande do sul, brasil",RS,novo hamburgo,novo hamburgo,RS,True
3,05e107217c7266362fd44b75b2cd4cc4,sao paulo - sp,SP,sao paulo,sao paulo,SP,True
4,4aba391bc3b88717ce08eb11e44937b2,arraial d'ajuda (porto seguro),BA,arraial d'ajuda,arraial d'ajuda,BA,True
5,8090490573c6c0aa343a7231ebcb8c86,sao paulo - sp,SP,sao paulo,sao paulo,SP,True
6,20cb7c2fde3e5bf10f0bbe7394e1c6a9,andira-pr,PR,andira,andira,PR,True
7,4b5f66b7adcf57f1ecc0d3c07dd6b177,vendas@creditparts.com.br,PR,vendas,NaN,NaN,False
8,06579cb253ecd5a3a12a9e6eb6bf8f47,sao paulo - sp,SP,sao paulo,sao paulo,SP,True
9,f9eedec3129e8cc6b6429c42d0808c5b,"rio de janeiro, rio de janeiro, brasil",RJ,rio de janeiro,rio de janeiro,RJ,True


In [53]:
# Investigate suspicious values that do not match a geolocation city

non_city_matches = city_component_check[
    ~city_component_check["city_component_match"]
].copy()

non_city_matches[
    [
        "seller_id",
        "seller_city",
        "seller_zip_code_prefix",
        "seller_state",
        "city_component"
    ]
]

,seller_id,seller_city,seller_zip_code_prefix,seller_state,city_component
1,ceb7b4fb9401cd378de7886317ad1b47,04482255,22790,RJ,04482255
7,4b5f66b7adcf57f1ecc0d3c07dd6b177,vendas@creditparts.com.br,87025,PR,vendas


In [54]:
# Compare non-city values against the geolocation reference using ZIP code and state

non_city_zip_check = (
    non_city_matches[
        [
            "seller_id",
            "seller_city",
            "seller_zip_code_prefix",
            "seller_state",
            "city_component"
        ]
    ]
    .copy()
)

non_city_zip_check["zip_prefix"] = (
    non_city_zip_check["seller_zip_code_prefix"]
    .astype(str)
    .str.zfill(5)
)

geolocation_zip_reference = (
    geolocation_df[
        [
            "geolocation_zip_code_prefix",
            "geolocation_city",
            "geolocation_state"
        ]
    ]
    .drop_duplicates()
    .copy()
)

geolocation_zip_reference["geolocation_zip_code_prefix"] = (
    geolocation_zip_reference["geolocation_zip_code_prefix"]
    .astype(str)
    .str.zfill(5)
)

non_city_zip_check = non_city_zip_check.merge(
    geolocation_zip_reference,
    left_on=[
        "zip_prefix",
        "seller_state"
    ],
    right_on=[
        "geolocation_zip_code_prefix",
        "geolocation_state"
    ],
    how="left"
)

non_city_zip_check[
    [
        "seller_id",
        "seller_city",
        "seller_zip_code_prefix",
        "seller_state",
        "geolocation_city",
        "geolocation_state"
    ]
]

,seller_id,seller_city,seller_zip_code_prefix,seller_state,geolocation_city,geolocation_state
0,ceb7b4fb9401cd378de7886317ad1b47,04482255,22790,RJ,rio de janeiro,RJ
1,4b5f66b7adcf57f1ecc0d3c07dd6b177,vendas@creditparts.com.br,87025,PR,maringa,PR


**Finding:** Both remaining non-city values were successfully matched to valid locations in the geolocation reference using the seller ZIP code and state. The numeric value `04482255` corresponds to **Rio de Janeiro, RJ**, while the email address `vendas@creditparts.com.br` corresponds to **Maringa, PR**. This confirms that the anomalies are confined to the `seller_city` field and do not indicate missing or invalid seller locations.

#### Seller -> Order Items referential integrity

In [55]:
order_items_df = pd.read_csv (r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_datasets\cleaned\olist_order_items_cleaned.csv")

In [56]:
# Validate seller_id referential integrity between Order Items and Sellers

order_item_seller_ids = set(order_items_df["seller_id"].dropna().unique())
seller_ids = set(sellers_df["seller_id"].dropna().unique())

missing_sellers = order_item_seller_ids - seller_ids
unused_sellers = seller_ids - order_item_seller_ids

print(f"Seller IDs in Order Items: {len(order_item_seller_ids):,}")
print(f"Seller IDs in Sellers: {len(seller_ids):,}")
print(f"Order Item seller IDs missing from Sellers: {len(missing_sellers):,}")
print(f"Sellers not referenced by Order Items: {len(unused_sellers):,}")


Seller IDs in Order Items: 3,095
Seller IDs in Sellers: 3,095
Order Item seller IDs missing from Sellers: 0
Sellers not referenced by Order Items: 0


 All 3,095 seller IDs in the Order Items dataset are present in the Sellers dataset, and every seller is referenced by at least one order item. No orphaned or unreferenced seller records were identified.

In [57]:
# Validate seller ZIP code and state against the Geolocation reference

seller_zip_geo_check = sellers_df[
    [
        "seller_id",
        "seller_zip_code_prefix",
        "seller_state"
    ]
].copy()

seller_zip_geo_check["zip_prefix"] = (
    seller_zip_geo_check["seller_zip_code_prefix"]
    .astype(str)
    .str.zfill(5)
)

geolocation_zip_reference = (
    geolocation_df[
        [
            "geolocation_zip_code_prefix",
            "geolocation_city",
            "geolocation_state"
        ]
    ]
    .drop_duplicates()
    .copy()
)

geolocation_zip_reference["geolocation_zip_code_prefix"] = (
    geolocation_zip_reference["geolocation_zip_code_prefix"]
    .astype(str)
    .str.zfill(5)
)

seller_zip_geo_check = seller_zip_geo_check.merge(
    geolocation_zip_reference,
    left_on=["zip_prefix", "seller_state"],
    right_on=[
        "geolocation_zip_code_prefix",
        "geolocation_state"
    ],
    how="left"
)

seller_zip_geo_check["geolocation_match"] = (
    seller_zip_geo_check["geolocation_city"].notna()
)

print(
    f"Sellers with a matching ZIP + state in Geolocation: "
    f"{seller_zip_geo_check['geolocation_match'].sum():,}"
)

print(
    f"Sellers without a matching ZIP + state in Geolocation: "
    f"{(~seller_zip_geo_check['geolocation_match']).sum():,}"
)

Sellers with a matching ZIP + state in Geolocation: 3,131
Sellers without a matching ZIP + state in Geolocation: 42


In [58]:
# Inspect sellers without a matching ZIP + state in Geolocation

unmatched_sellers = seller_zip_geo_check[
    ~seller_zip_geo_check["geolocation_match"]
].copy()

unmatched_sellers[
    [
        "seller_id",
        "seller_zip_code_prefix",
        "zip_prefix",
        "seller_state"
    ]
].drop_duplicates()

,seller_id,seller_zip_code_prefix,zip_prefix,seller_state
74,f410c8873029fcc3809b9df6d0b28914,95076,95076,SP
119,392f7f2c797e4dc077e4311bde2ab8ce,21210,21210,RN
205,1284de4ae8aa26997e748c851557cf0e,85301,85301,SP
213,8b181ee5518df84f18f4e1a43fe07923,87360,87360,SP
283,3d700782d7818f2c1e0d7a9e9d75fc00,86170,86170,SP
319,f626e15b7314c267e4429010866f70e9,85960,85960,SP
374,c716e0b86ed568878475b60fbb6323ad,22783,22783,SP
434,c8771b1a10bb99bb34d3c459c5cffb53,36512,36512,SP
483,5962468f885ea01a1b6a97a218797b0a,82040,82040,PR
527,48436dade18ac8b2bce089ec2a041202,27277,27277,SP


In [59]:
geolocation_df = pd.read_csv (r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce\01_datasets\cleaned\olist_geolocation_cleaned.csv")

In [60]:
# Check whether unmatched ZIP prefixes exist in Geolocation under another state

unmatched_zip_check = unmatched_sellers[
    ["seller_id", "seller_zip_code_prefix", "zip_prefix", "seller_state"]
].drop_duplicates().merge(
    geolocation_zip_reference[
        [
            "geolocation_zip_code_prefix",
            "geolocation_state"
        ]
    ].drop_duplicates(),
    left_on="zip_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left"
)

unmatched_zip_check[
    [
        "seller_id",
        "seller_zip_code_prefix",
        "zip_prefix",
        "seller_state",
        "geolocation_state"
    ]
]

,seller_id,seller_zip_code_prefix,zip_prefix,seller_state,geolocation_state
0,f410c8873029fcc3809b9df6d0b28914,95076,95076,SP,RS
1,392f7f2c797e4dc077e4311bde2ab8ce,21210,21210,RN,RJ
2,1284de4ae8aa26997e748c851557cf0e,85301,85301,SP,PR
3,8b181ee5518df84f18f4e1a43fe07923,87360,87360,SP,PR
4,3d700782d7818f2c1e0d7a9e9d75fc00,86170,86170,SP,PR
5,f626e15b7314c267e4429010866f70e9,85960,85960,SP,PR
6,c716e0b86ed568878475b60fbb6323ad,22783,22783,SP,RJ
7,c8771b1a10bb99bb34d3c459c5cffb53,36512,36512,SP,MG
8,5962468f885ea01a1b6a97a218797b0a,82040,82040,PR,NaN
9,48436dade18ac8b2bce089ec2a041202,27277,27277,SP,RJ


- The Seller ZIP exists in the Geolocation reference, but **the state attached to that ZIP does not match** `seller_state` for many records.
- Some ZIPs have a Geolocation match under a different state, e.g. seller SP vs Geolocation `PR`, `RS`, `RJ`, etc.
- Some have no **Geolocation state at all (NaN)**.
- Therefore, the original 42 unmatched records are not 42 missing ZIP codes. They're 42 cases where the ZIP + Seller State combination wasn't found.

In [61]:
# Classify unmatched seller ZIP + state combinations

unmatched_zip_check["zip_exists_in_geolocation"] = (
    unmatched_zip_check["geolocation_state"].notna()
)

same_state_count = (
    unmatched_zip_check["geolocation_state"]
    == unmatched_zip_check["seller_state"]
).sum()

different_state_count = (
    unmatched_zip_check["geolocation_state"].notna()
    & (
        unmatched_zip_check["geolocation_state"]
        != unmatched_zip_check["seller_state"]
    )
).sum()

missing_zip_count = (
    unmatched_zip_check["geolocation_state"].isna()
).sum()

print(f"ZIP + state matches: {same_state_count}")
print(f"ZIP exists but state differs: {different_state_count}")
print(f"ZIP not represented in Geolocation: {missing_zip_count}")

ZIP + state matches: 0
ZIP exists but state differs: 35
ZIP not represented in Geolocation: 7


In [62]:
# Investigate the 42 unmatched ZIP + state combinations using seller city

seller_geo_exception_check = (
    unmatched_sellers[
        [
            "seller_id",
            "seller_zip_code_prefix",
            "seller_state",
            "zip_prefix"
        ]
    ]
    .drop_duplicates()
    .merge(
        sellers_df[
            [
                "seller_id",
                "seller_city"
            ]
        ],
        on="seller_id",
        how="left"
    )
    .merge(
        geolocation_df[
            [
                "geolocation_zip_code_prefix",
                "geolocation_city",
                "geolocation_state"
            ]
        ]
        .assign(
            geolocation_zip_code_prefix=lambda x:
                x["geolocation_zip_code_prefix"]
                .astype(str)
                .str.zfill(5)
        )
        .drop_duplicates(),
        left_on="zip_prefix",
        right_on="geolocation_zip_code_prefix",
        how="left"
    )
)

seller_geo_exception_check[
    [
        "seller_id",
        "seller_city",
        "seller_zip_code_prefix",
        "seller_state",
        "geolocation_city",
        "geolocation_state"
    ]
]

,seller_id,seller_city,seller_zip_code_prefix,seller_state,geolocation_city,geolocation_state
0,f410c8873029fcc3809b9df6d0b28914,caxias do sul,95076,SP,caxias do sul,RS
1,392f7f2c797e4dc077e4311bde2ab8ce,rio de janeiro,21210,RN,rio de janeiro,RJ
2,1284de4ae8aa26997e748c851557cf0e,laranjeiras do sul,85301,SP,laranjeiras do sul,PR
3,8b181ee5518df84f18f4e1a43fe07923,goioere,87360,SP,goioere,PR
4,3d700782d7818f2c1e0d7a9e9d75fc00,sertanopolis,86170,SP,sertanopolis,PR
5,f626e15b7314c267e4429010866f70e9,marechal candido rondon,85960,SP,marechal candido rondon,PR
6,c716e0b86ed568878475b60fbb6323ad,rio de janeiro,22783,SP,rio de janeiro,RJ
7,c8771b1a10bb99bb34d3c459c5cffb53,tocantins,36512,SP,tocantins,MG
8,5962468f885ea01a1b6a97a218797b0a,curitiba,82040,PR,NaN,NaN
9,48436dade18ac8b2bce089ec2a041202,volta redonda,27277,SP,volta redonda,RJ


The seller ZIP code prefixes were standardized to a five-character string format before comparison with the Geolocation dataset. This step ensures that prefixes originally stored as integers, such as `2285`, are correctly represented as `02285` and compared consistently with the Geolocation ZIP prefixes.

The comparison identified **42 seller records without an exact ZIP + state match** in the Geolocation dataset. Further investigation showed that **35 ZIP prefixes were present in the Geolocation dataset but were associated with a different state, while 7 ZIP prefixes were not represented in the Geolocation reference at all**.

In [63]:
# Investigate unmatched ZIP + state combinations using seller city

# Standardize seller ZIP prefixes
seller_zip_state_exceptions = (
    sellers_df[
        [
            "seller_id",
            "seller_city",
            "seller_zip_code_prefix",
            "seller_state"
        ]
    ]
    .drop_duplicates()
    .copy()
)

seller_zip_state_exceptions["zip_prefix"] = (
    seller_zip_state_exceptions["seller_zip_code_prefix"]
    .astype(str)
    .str.zfill(5)
)

# Create unique ZIP + state combinations from Geolocation
geolocation_zip_state_reference = (
    geolocation_df[
        [
            "geolocation_zip_code_prefix",
            "geolocation_state"
        ]
    ]
    .drop_duplicates()
    .copy()
)

geolocation_zip_state_reference["geolocation_zip_code_prefix"] = (
    geolocation_zip_state_reference["geolocation_zip_code_prefix"]
    .astype(str)
    .str.zfill(5)
)

# Identify seller ZIP + state combinations not represented in Geolocation
seller_zip_state_exceptions = (
    seller_zip_state_exceptions
    .merge(
        geolocation_zip_state_reference,
        left_on=["zip_prefix", "seller_state"],
        right_on=["geolocation_zip_code_prefix", "geolocation_state"],
        how="left",
        indicator=True
    )
    .query("_merge == 'left_only'")
    .drop(
        columns=[
            "geolocation_zip_code_prefix",
            "geolocation_state",
            "_merge"
        ]
    )
)

# Compare the exception ZIP prefixes against Geolocation using ZIP only
geolocation_zip_reference = (
    geolocation_df[
        [
            "geolocation_zip_code_prefix",
            "geolocation_city",
            "geolocation_state"
        ]
    ]
    .drop_duplicates()
    .copy()
)

geolocation_zip_reference["geolocation_zip_code_prefix"] = (
    geolocation_zip_reference["geolocation_zip_code_prefix"]
    .astype(str)
    .str.zfill(5)
)

seller_zip_state_city_check = (
    seller_zip_state_exceptions
    .merge(
        geolocation_zip_reference,
        left_on="zip_prefix",
        right_on="geolocation_zip_code_prefix",
        how="left"
    )
)

seller_zip_state_city_check[
    [
        "seller_id",
        "seller_city",
        "seller_zip_code_prefix",
        "seller_state",
        "geolocation_city",
        "geolocation_state"
    ]
].drop_duplicates()

,seller_id,seller_city,seller_zip_code_prefix,seller_state,geolocation_city,geolocation_state
0,f410c8873029fcc3809b9df6d0b28914,caxias do sul,95076,SP,caxias do sul,RS
1,392f7f2c797e4dc077e4311bde2ab8ce,rio de janeiro,21210,RN,rio de janeiro,RJ
2,1284de4ae8aa26997e748c851557cf0e,laranjeiras do sul,85301,SP,laranjeiras do sul,PR
3,8b181ee5518df84f18f4e1a43fe07923,goioere,87360,SP,goioere,PR
4,3d700782d7818f2c1e0d7a9e9d75fc00,sertanopolis,86170,SP,sertanopolis,PR
5,f626e15b7314c267e4429010866f70e9,marechal candido rondon,85960,SP,marechal candido rondon,PR
6,c716e0b86ed568878475b60fbb6323ad,rio de janeiro,22783,SP,rio de janeiro,RJ
7,c8771b1a10bb99bb34d3c459c5cffb53,tocantins,36512,SP,tocantins,MG
8,5962468f885ea01a1b6a97a218797b0a,curitiba,82040,PR,NaN,NaN
9,48436dade18ac8b2bce089ec2a041202,volta redonda,27277,SP,volta redonda,RJ


In [64]:
# Final classification of seller ZIP + state exceptions

# Standardize ZIP prefixes without altering the original columns
seller_zip_check = (
    sellers_df[
        [
            "seller_id",
            "seller_city",
            "seller_zip_code_prefix",
            "seller_state"
        ]
    ]
    .drop_duplicates()
    .copy()
)

seller_zip_check["zip_prefix"] = (
    seller_zip_check["seller_zip_code_prefix"]
    .astype(str)
    .str.zfill(5)
)

geo_zip_check = (
    geolocation_df[
        [
            "geolocation_zip_code_prefix",
            "geolocation_city",
            "geolocation_state"
        ]
    ]
    .drop_duplicates()
    .copy()
)

geo_zip_check["zip_prefix"] = (
    geo_zip_check["geolocation_zip_code_prefix"]
    .astype(str)
    .str.zfill(5)
)

# Identify whether each seller ZIP exists in Geolocation
seller_zip_check["zip_exists_in_geolocation"] = (
    seller_zip_check["zip_prefix"]
    .isin(geo_zip_check["zip_prefix"])
)

# Identify whether the ZIP + state combination exists
zip_state_reference = (
    geo_zip_check[
        ["zip_prefix", "geolocation_state"]
    ]
    .drop_duplicates()
)

seller_zip_check["zip_state_matches"] = (
    seller_zip_check
    .merge(
        zip_state_reference,
        left_on=["zip_prefix", "seller_state"],
        right_on=["zip_prefix", "geolocation_state"],
        how="left",
        indicator=True
    )["_merge"]
    .eq("both")
)

# Keep only the previously identified ZIP + state exceptions
seller_zip_state_exceptions = seller_zip_check[
    ~seller_zip_check["zip_state_matches"]
].copy()

# Classify the exceptions
seller_zip_state_exceptions["exception_type"] = (
    seller_zip_state_exceptions["zip_exists_in_geolocation"]
    .map({
        True: "ZIP exists but state differs",
        False: "ZIP not represented in Geolocation"
    })
)

# Final evidence table
seller_zip_state_exceptions[
    [
        "seller_id",
        "seller_city",
        "seller_zip_code_prefix",
        "zip_prefix",
        "seller_state",
        "exception_type"
    ]
].sort_values(
    ["exception_type", "seller_id"]
).reset_index(drop=True)

,seller_id,seller_city,seller_zip_code_prefix,zip_prefix,seller_state,exception_type
0,0bae85eb84b9fb3bd773911e89288d54,itajai,88301,88301,SP,ZIP exists but state differs
1,1284de4ae8aa26997e748c851557cf0e,laranjeiras do sul,85301,85301,SP,ZIP exists but state differs
2,20a7efa9721046319bdde5d60b6b5365,laguna,88790,88790,SP,ZIP exists but state differs
3,289cdb325fb7e7f891c38608bf9e0962,belo horizonte,31570,31570,SP,ZIP exists but state differs
4,296729ffb9b684050dd24836dac4494a,curitiba,80240,80240,SP,ZIP exists but state differs
5,2a167ca73899c85001a837d8fb4962f6,sao paulo,37540,37540,SP,ZIP exists but state differs
6,392f7f2c797e4dc077e4311bde2ab8ce,rio de janeiro,21210,21210,RN,ZIP exists but state differs
7,3a52d63a8f9daf5a28f3626d7eb9bd28,aguas claras df,71900,71900,SP,ZIP exists but state differs
8,3d700782d7818f2c1e0d7a9e9d75fc00,sertanopolis,86170,86170,SP,ZIP exists but state differs
9,48436dade18ac8b2bce089ec2a041202,volta redonda,27277,27277,SP,ZIP exists but state differs


In [65]:
print(
    "Total ZIP + state exceptions:",
    len(seller_zip_state_exceptions)
)

print(
    "ZIP exists but state differs:",
    (
        seller_zip_state_exceptions["exception_type"]
        == "ZIP exists but state differs"
    ).sum()
)

print(
    "ZIP not represented in Geolocation:",
    (
        seller_zip_state_exceptions["exception_type"]
        == "ZIP not represented in Geolocation"
    ).sum()
)

Total ZIP + state exceptions: 42
ZIP exists but state differs: 35
ZIP not represented in Geolocation: 7


Seller ZIP code prefixes were standardized to a five-digit string format for comparison while preserving the original seller values. The seller ZIP + state combinations were then compared against the Geolocation reference dataset.

A total of **42 seller records** were identified where the ZIP + state combination was not represented in Geolocation. Further investigation showed that **35 records had ZIP prefixes that existed in Geolocation but were associated with a different state**, while **7 ZIP prefixes were not represented in the Geolocation dataset at all**.

These results indicate that the exceptions are primarily **ZIP + state inconsistencies rather than formatting issues**. The original seller values were retained and no automatic corrections were applied because Geolocation was used as a reference for investigation rather than as sufficient evidence to overwrite seller data.

### **3. Data Cleaning**

##### ZIP Code Prefix

In [66]:
# Clean seller ZIP code prefixes and convert to sting

sellers_cleaned = sellers_df.copy()

sellers_cleaned["seller_zip_code_prefix"] = (
    sellers_cleaned["seller_zip_code_prefix"]
    .astype(str)
    .str.zfill(5)
)

In [67]:
# Convert seller ZIP code prefix to a standardized 5-digit string

sellers_cleaned["seller_zip_code_prefix"] = (
    sellers_cleaned["seller_zip_code_prefix"]
    .astype(str)
    .str.zfill(5)
)

In [68]:
# Validate ZIP code data type and format

print(
    "ZIP code data type:",
    sellers_cleaned["seller_zip_code_prefix"].dtype
)

print(
    "ZIP codes not exactly 5 characters:",
    (
        sellers_cleaned["seller_zip_code_prefix"].str.len() != 5
    ).sum()
)

print(
    "ZIP codes containing non-numeric characters:",
    (
        ~sellers_cleaned["seller_zip_code_prefix"].str.isdigit()
    ).sum()
)

ZIP code data type: object
ZIP codes not exactly 5 characters: 0
ZIP codes containing non-numeric characters: 0


The `seller_zip_code_prefix` column was converted from an integer to a string and standardized to a five-character format by restoring omitted leading zeros. This preserves ZIP code prefixes as geographic identifiers rather than numerical values.

##### City Whitespace

In [69]:
# Clean seller city whitespace

sellers_cleaned["seller_city"] = (
    sellers_cleaned["seller_city"]
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

In [70]:
# Validate seller city whitespace cleaning

leading_trailing_whitespace = (
    sellers_cleaned["seller_city"]
    .str.contains(r"^\s|\s$", regex=True, na=False)
).sum()

consecutive_whitespace = (
    sellers_cleaned["seller_city"]
    .str.contains(r"\s{2,}", regex=True, na=False)
).sum()

print(
    "Seller city records with leading/trailing whitespace:",
    leading_trailing_whitespace
)

print(
    "Seller city records with consecutive whitespace:",
    consecutive_whitespace
)

Seller city records with leading/trailing whitespace: 0
Seller city records with consecutive whitespace: 0


Leading and trailing whitespace was removed from `seller_city`, and consecutive whitespace characters were normalized to a single space. 

In [71]:
import unicodedata

# Normalize Unicode representation in seller city values

sellers_cleaned["seller_city"] = (
    sellers_cleaned["seller_city"]
    .apply(lambda x: unicodedata.normalize("NFC", x) if isinstance(x, str) else x)
)

In [72]:
# Validate that all seller city values are NFC-normalized

unicode_not_normalized = (
    sellers_cleaned["seller_city"]
    .apply(
        lambda x: (
            isinstance(x, str)
            and x != unicodedata.normalize("NFC", x)
        )
    )
    .sum()
)

print(
    "Seller city records not in NFC-normalized form:",
    unicode_not_normalized
)

Seller city records not in NFC-normalized form: 0


Unicode normalization was applied to `seller_city` using NFC normalization to ensure that visually equivalent characters are represented consistently. This standardizes character encoding without removing legitimate accents or punctuation from city names.

##### Remove verified state/country suffixes

In [73]:
# Remove verified state/country information from seller city values

slash_mask = sellers_cleaned["seller_city"].str.contains(
    r"[/\\]",
    regex=True,
    na=False
)

sellers_cleaned.loc[slash_mask, "seller_city"] = (
    sellers_cleaned.loc[slash_mask, "seller_city"]
    .str.split(r"[/\\]", regex=True)
    .str[0]
    .str.strip()
)

In [74]:
# Validate slash/backslash cleaning

remaining_slash_values = (
    sellers_cleaned["seller_city"]
    .str.contains(r"[/\\]", regex=True, na=False)
    .sum()
)

print(
    "Seller city records still containing forward/backward slashes:",
    remaining_slash_values
)

Seller city records still containing forward/backward slashes: 0


The previously identified seller city values containing forward or backward slashes were cleaned using the city component established during profiling. Redundant state or geographic information following the separator was removed, while the verified city component was retained.

##### Clean `seller_state`

In [75]:
# Standardize seller state formatting

sellers_cleaned["seller_state"] = (
    sellers_cleaned["seller_state"]
    .astype(str)
    .str.strip()
    .str.upper()
)

In [76]:
# Validate seller state formatting

print(
    "Seller state values with leading/trailing whitespace:",
    sellers_cleaned["seller_state"]
    .str.contains(r"^\s|\s$", regex=True, na=False)
    .sum()
)

print(
    "Seller state values not in uppercase:",
    (
        sellers_cleaned["seller_state"]
        != sellers_cleaned["seller_state"].str.upper()
    ).sum()
)

print(
    "Missing seller state values:",
    sellers_cleaned["seller_state"].isna().sum()
)

Seller state values with leading/trailing whitespace: 0
Seller state values not in uppercase: 0
Missing seller state values: 0


##### Apply accent standardization

In [77]:
from unidecode import unidecode

# Standardize accented seller city names to unaccented forms

sellers_cleaned["seller_city"] = (
    sellers_cleaned["seller_city"]
    .apply(lambda x: unidecode(x) if isinstance(x, str) else x)
)

In [78]:
# Validate accent standardization

remaining_non_ascii = (
    sellers_cleaned["seller_city"]
    .apply(
        lambda x: (
            isinstance(x, str)
            and any(ord(char) > 127 for char in x)
        )
    )
    .sum()
)

print(
    "Seller city records containing non-ASCII characters after accent standardization:",
    remaining_non_ascii
)

Seller city records containing non-ASCII characters after accent standardization: 0


In [79]:
# Confirm legitimate punctuation remains after accent standardization

punctuation_examples = sellers_cleaned[
    sellers_cleaned["seller_city"].str.contains(
        r"['()\-]",
        regex=True,
        na=False
    )
][
    ["seller_id", "seller_city", "seller_state"]
]

punctuation_examples.head(20)

,seller_id,seller_city,seller_state
78,731ef20c231d9a7103a425e83fd91271,lages - sc,SC
476,26b482dccfa29bd2e40703ba45523702,santa barbara d'oeste,SP
707,11938fcc75f6902fea1d0a6f72b54a57,sao miguel d'oeste,SC
826,05e107217c7266362fd44b75b2cd4cc4,sao paulo - sp,SP
874,4aba391bc3b88717ce08eb11e44937b2,arraial d'ajuda (porto seguro),BA
1580,8090490573c6c0aa343a7231ebcb8c86,sao paulo - sp,SP
1610,20cb7c2fde3e5bf10f0bbe7394e1c6a9,andira-pr,PR
1890,30a2f535bb48308f991d0b9ad4a8c4bb,santa barbara d'oeste,SP
1919,13dd308f81fea30cc670c656b2b46cc3,santa barbara d'oeste,SP
1982,20f0aeea30bc3b8c4420be8ced4226c0,santa barbara d'oeste,SP


Accented characters in `seller_city` were standardized to their unaccented equivalents using `unidecode()`, consistent with the standardization approach applied to related datasets. This transformation affects accented characters only; legitimate punctuation such as apostrophes, hyphens, and parentheses was intentionally retained.

In [80]:
sellers_df.duplicated().sum()

np.int64(0)

#### **Additional Decision**

**Non-City Seller Values**

Two `seller_city` values were identified during profiling as containing non-city information: a numeric ZIP code (`04482255`) and an email address (`vendas@creditparts.com.br`). Their corresponding ZIP + state combinations were found in the Geolocation dataset, providing geographic reference information. However, the original seller city values were retained because replacing source values with inferred city names would constitute an unsupported correction. These records were therefore not modified during cleaning.

**ZIP + State Discrepancies**

A total of 42 seller ZIP + state combinations were not represented as matching combinations in the Geolocation dataset. Of these, 35 ZIP prefixes existed in Geolocation but were associated with a different state, while 7 ZIP prefixes were not represented in Geolocation. These values were not corrected because Geolocation was used as a reference for consistency checking rather than sufficient evidence to overwrite the original seller ZIP or state values. The original source values were therefore retained.

**Legitimate City Punctuation**

Punctuation appearing within legitimate city names was not standardized or removed unless it was part of a separate, explicitly identified cleaning decision. Apostrophes, hyphens, parentheses, and other legitimate punctuation were retained to avoid unnecessarily altering valid city-name information.

**Geographic Reassignment**

Seller geographic values were not reassigned solely to make them agree with the Geolocation dataset. Differences between seller ZIP/state values and Geolocation were treated as cross-dataset consistency exceptions rather than automatic corrections. Original seller values were retained where the profiling stage did not establish an objectively supported replacement.

**Ambiguous City Representations**

City values with multiple geographic references or representations were not automatically replaced with a value inferred from the Geolocation dataset. Where the profiling evidence did not establish a single unambiguous city value, the original seller value was retained rather than introducing an assumption during cleaning.



In [82]:
# Final validation of cleaned Sellers dataset


# 1. Dataset integrity
print("1. DATASET INTEGRITY")
print("Rows preserved:", len(sellers_cleaned) == len(sellers_df))
print("Expected rows:", len(sellers_df))
print("Columns preserved:", list(sellers_cleaned.columns) == list(sellers_df.columns))
print("Missing values:", sellers_cleaned.isna().sum().sum())
print(
    "Duplicate seller IDs:",
    sellers_cleaned["seller_id"].duplicated().sum()
)

# 2. Seller ID validation
print("\n2. SELLER ID")
print(
    "Invalid seller ID length:",
    (
        sellers_cleaned["seller_id"].astype(str).str.len() != 32
    ).sum()
)
print(
    "Seller IDs with invalid characters:",
    (
        ~sellers_cleaned["seller_id"]
        .astype(str)
        .str.fullmatch(r"[A-Za-z0-9]+")
    ).sum()
)

# 3. ZIP code validation
print("\n3. ZIP CODE PREFIX")
print(
    "ZIP code data type:",
    sellers_cleaned["seller_zip_code_prefix"].dtype
)
print(
    "ZIP codes not exactly 5 characters:",
    (
        sellers_cleaned["seller_zip_code_prefix"]
        .astype(str)
        .str.len() != 5
    ).sum()
)
print(
    "ZIP codes containing non-numeric characters:",
    (
        ~sellers_cleaned["seller_zip_code_prefix"]
        .astype(str)
        .str.fullmatch(r"\d{5}")
    ).sum()
)

# 4. Seller city validation
print("\n4. SELLER CITY")
print(
    "Leading/trailing whitespace:",
    sellers_cleaned["seller_city"]
    .str.contains(r"^\s|\s$", regex=True, na=False)
    .sum()
)
print(
    "Consecutive whitespace:",
    sellers_cleaned["seller_city"]
    .str.contains(r"\s{2,}", regex=True, na=False)
    .sum()
)
print(
    "Non-NFC Unicode values:",
    sellers_cleaned["seller_city"]
    .apply(
        lambda x: (
            isinstance(x, str)
            and x != unicodedata.normalize("NFC", x)
        )
    )
    .sum()
)
print(
    "Remaining non-ASCII city values:",
    sellers_cleaned["seller_city"]
    .apply(
        lambda x: (
            isinstance(x, str)
            and any(ord(char) > 127 for char in x)
        )
    )
    .sum()
)
print(
    "Remaining slash/backslash values:",
    sellers_cleaned["seller_city"]
    .str.contains(r"[/\\]", regex=True, na=False)
    .sum()
)

# 5. Seller state validation
print("\n5. SELLER STATE")
print(
    "State values with whitespace:",
    sellers_cleaned["seller_state"]
    .str.contains(r"^\s|\s$", regex=True, na=False)
    .sum()
)
print(
    "State values not uppercase:",
    (
        sellers_cleaned["seller_state"]
        != sellers_cleaned["seller_state"].str.upper()
    ).sum()
)

# 6. Final overall status
validation_checks = {
    "Row count preserved":
        len(sellers_cleaned) == len(sellers_df),

    "Columns preserved":
        list(sellers_cleaned.columns) == list(sellers_df.columns),

    "No missing values":
        sellers_cleaned.isna().sum().sum() == 0,

    "Seller IDs unique":
        sellers_cleaned["seller_id"].duplicated().sum() == 0,

    "Seller IDs valid length":
        (sellers_cleaned["seller_id"].astype(str).str.len() == 32).all(),

    "ZIP codes exactly 5 digits":
        sellers_cleaned["seller_zip_code_prefix"]
        .astype(str)
        .str.fullmatch(r"\d{5}")
        .all(),

    "No city whitespace issues":
        (
            ~sellers_cleaned["seller_city"]
            .str.contains(r"^\s|\s|\s{2,}", regex=True, na=False)
        ).all(),

    "Cities NFC normalized":
        (
            sellers_cleaned["seller_city"]
            .apply(
                lambda x: (
                    not isinstance(x, str)
                    or x == unicodedata.normalize("NFC", x)
                )
            )
        ).all(),

    "No remaining slash/backslash contamination":
        (
            ~sellers_cleaned["seller_city"]
            .str.contains(r"[/\\]", regex=True, na=False)
        ).all(),

    "States uppercase":
        (
            sellers_cleaned["seller_state"]
            == sellers_cleaned["seller_state"].str.upper()
        ).all()
}



sellers_cleaned.info()



1. DATASET INTEGRITY
Rows preserved: True
Expected rows: 3095
Columns preserved: True
Missing values: 0
Duplicate seller IDs: 0

2. SELLER ID
Invalid seller ID length: 0
Seller IDs with invalid characters: 0

3. ZIP CODE PREFIX
ZIP code data type: object
ZIP codes not exactly 5 characters: 0
ZIP codes containing non-numeric characters: 0

4. SELLER CITY
Leading/trailing whitespace: 0
Consecutive whitespace: 0
Non-NFC Unicode values: 0
Remaining non-ASCII city values: 0
Remaining slash/backslash values: 0

5. SELLER STATE
State values with whitespace: 0
State values not uppercase: 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3095 entries, 0 to 3094
Data columns (total 4 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   seller_id               3095 non-null   object
 1   seller_zip_code_prefix  3095 non-null   object
 2   seller_city             3095 non-null   object
 3   seller_state            3095 non-null   

### **4. Export The Cleaned Dataset**

In [83]:
PROJECT_ROOT = Path(r"C:\Users\Deviare User\OneDrive\Desktop\e-commerce")

RAW_DATA_PATH = PROJECT_ROOT / "01_datasets" / "raw"
CLEANED_DATA_PATH = PROJECT_ROOT / "01_datasets" / "cleaned"

In [84]:
sellers_cleaned.to_csv(
    CLEANED_DATA_PATH / "olist_sellers_cleaned.csv",
    index=False
)